# YouTube Chatbot with Groq API
This notebook fetches YouTube transcripts and answers questions using Groq LLM

In [ ]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from youtube_transcript_api import YouTubeTranscriptApi
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document

In [ ]:
# Load API key
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if GROQ_API_KEY:
    print("✅ API Key loaded successfully!")
else:
    print("❌ API Key NOT found. Check your .env file.")

In [ ]:
# Fetch YouTube transcript
VIDEO_ID = "VGFpV3Qj4as"  # Extracted from v=VGFpV3Qj4as&t=1s

try:
    transcript_list = YouTubeTranscriptApi.get_transcript(VIDEO_ID)
    transcript = " ".join([t["text"] for t in transcript_list])
    print(f"✅ Transcript fetched! Length: {len(transcript)} characters")
    print(f"\nFirst 300 characters:\n{transcript[:300]}...")
except Exception as e:
    print(f"❌ Error: {e}")

In [ ]:
# Split text and create vector store
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_text(transcript)
documents = [Document(page_content=chunk) for chunk in chunks]

print(f"📄 Created {len(chunks)} text chunks")

# Create embeddings and vector store
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents, embeddings)

print("✅ Vector store created!")

In [ ]:
# Initialize Groq LLM and QA chain
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.1-70b-versatile",
    temperature=0.3
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=False
)

print("✅ QA Chain ready!")

In [ ]:
# Ask questions about the video
question = "What is this video about?"  # Change your question here

response = qa_chain.invoke({"query": question})
print(f"Question: {question}")
print(f"\nAnswer: {response['result']}")

In [ ]:
# Interactive chat (run this cell multiple times with different questions)
question = input("Ask a question: ")
response = qa_chain.invoke({"query": question})
print(f"\n🤖 Answer: {response['result']}")